In [1]:
# =========================================
# CHILD GALLOPING TEST (DATA-DRIVEN)
# =========================================

import cv2
import numpy as np
import mediapipe as mp
from scipy.signal import find_peaks
from ultralytics import YOLO

In [2]:
# -------------------------------
# MediaPipe Setup
# -------------------------------
mpPose = mp.solutions.pose
pose = mpPose.Pose(min_detection_confidence=0.6)

mpDraw = mp.solutions.drawing_utils
DRAW_LM = mpDraw.DrawingSpec(color=(0,0,255), thickness=2, circle_radius=2)
DRAW_CONN = mpDraw.DrawingSpec(color=(0,0,0), thickness=2)

In [3]:
# -------------------------------
# Utils
# -------------------------------
def safe_mean(x): return float(np.mean(x)) if len(x) else 0.0
def safe_std(x): return float(np.std(x)) if len(x) else 0.0

def smooth(x, k=5):
    if len(x) < k: return np.array(x)
    return np.convolve(x, np.ones(k)/k, mode='same')

def angle(a,b,c):
    a,b,c = np.array(a),np.array(b),np.array(c)
    ba, bc = a-b, c-b
    cos = np.dot(ba,bc)/(np.linalg.norm(ba)*np.linalg.norm(bc)+1e-6)
    return np.degrees(np.arccos(np.clip(cos,-1,1)))

In [4]:
# Lead Leg Consistency
def lead_score_cal(switches):
    lead_score = 0
    if switches == 0: lead_score = 5
    elif switches == 1: lead_score = 4
    elif switches <= 3: lead_score = 3
    elif switches <= 6: lead_score = 2
    else: lead_score = 1
    return lead_score

In [5]:
# Rhythm
def rhythm_score_cal(gallop_count, rhythm_var):
    rhythm_score = 0
    if gallop_count >= 4 and rhythm_var < 0.1: rhythm_score = 5
    elif gallop_count >= 3 and rhythm_var < 0.15: rhythm_score = 4
    elif gallop_count >= 2: rhythm_score = 3
    elif gallop_count >= 1: rhythm_score = 2
    else: rhythm_score = 1
    return rhythm_score

In [6]:
# Flight Phase
def flight_score_cal(flight_ratio):
    flight_score = 0
    if flight_ratio > 0.6: flight_score = 5
    elif flight_ratio > 0.45: flight_score = 4
    elif flight_ratio > 0.3: flight_score = 3
    elif flight_ratio > 0.1: flight_score = 2
    else: flight_score = 1
    return flight_score

In [7]:
# Arm Position
def arm_score_cal(arm_ratio):
    arm_score = 0
    if arm_ratio > 0.8: arm_score = 5
    elif arm_ratio > 0.6: arm_score = 4
    elif arm_ratio > 0.4: arm_score = 3
    elif arm_ratio > 0.2: arm_score = 2
    else: arm_score = 1
    return arm_score

In [11]:
def threshold_line(path):
    
    model = YOLO("best.pt")
    cap = cv2.VideoCapture(path)

    left_x, right_x = [], []
    frame_count = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        results = model(frame, conf=0.5, verbose=False)

        centers = []
        for box in results[0].boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = box.conf.item()
            cx = (x1 + x2) // 2
            centers.append(cx)
            
            # Draw rectangle on cone
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

            # Optional: confidence label
            cv2.putText(frame, f"Cone {conf:.2f}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

        if len(centers) >= 2:
            centers.sort()
            left_x.append(centers[0])
            right_x.append(centers[-1])

        if len(left_x) > 50:
            break

        cv2.namedWindow("Video", cv2.WINDOW_NORMAL)
        cv2.setWindowProperty("Video", cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)
        cv2.imshow("Video", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

    # Final fixed threshold
    left_avg = int(np.mean(left_x))
    right_avg = int(np.mean(right_x))
    # threshold_x = int((left_avg + right_avg) / 2)

    print("Left cone X:", left_avg)
    print("Right cone X:", right_avg)
    # print("Threshold X:", threshold_x)
    return left_avg, right_avg


In [9]:
# MAIN FUNCTION
# -------------------------------
def galloping_test(path="video.mp4"):

    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS)

    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    #threshold line calculate  
    left_line_x, right_line_x = 0.10, 0.85
    left_line_x, right_line_x = threshold_line(path)
    left_line_x = round((left_line_x/frame_width), 2) 
    right_line_x = round(((right_line_x)/frame_width), 2) 
    print(f"{left_line_x} --- {right_line_x}")

    # signals
    l_ank_y, r_ank_y = [], []
    l_knee_y, r_knee_y = [], []
    l_el_y, r_el_y = [], []
    l_sh_y, r_sh_y = [], []
    l_el_ang, r_el_ang = [], []

    frame_idx = 0
    window = "Galloping Analysis"
    cv2.namedWindow(window, cv2.WINDOW_NORMAL)

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        res = pose.process(rgb)

        if res.pose_landmarks:
            lms = res.pose_landmarks.landmark

            # hip center
            hx = (lms[23].x + lms[24].x) / 2

            # boundary check
            if not (left_line_x <= hx <= right_line_x):
                continue

            # visibility check (hip)
            if lms[23].visibility < 0.5:
                cv2.imshow(window, frame)
                if cv2.waitKey(1) & 0xFF == 27: break
                frame_idx += 1
                continue

            mpDraw.draw_landmarks(frame, res.pose_landmarks, mpPose.POSE_CONNECTIONS, DRAW_LM, DRAW_CONN)

            # keypoints
            l_sh=(lms[11].x,lms[11].y); r_sh=(lms[12].x,lms[12].y)
            l_el=(lms[13].x,lms[13].y); r_el=(lms[14].x,lms[14].y)
            l_hip=(lms[23].x,lms[23].y); r_hip=(lms[24].x,lms[24].y)
            l_knee=(lms[25].x,lms[25].y); r_knee=(lms[26].x,lms[26].y)
            l_ank=(lms[27].x,lms[27].y); r_ank=(lms[28].x,lms[28].y)
            l_wri=(lms[15].x,lms[15].y); r_wri=(lms[16].x,lms[16].y)

            # signals
            l_ank_y.append(l_ank[1]); r_ank_y.append(r_ank[1])
            l_knee_y.append(l_knee[1]); r_knee_y.append(r_knee[1])
            l_el_y.append(l_el[1]); r_el_y.append(r_el[1])
            l_sh_y.append(l_sh[1]); r_sh_y.append(r_sh[1])

            # elbow angle (arm bend)
            l_el_ang.append(angle(l_sh, l_el, l_wri))
            r_el_ang.append(angle(r_sh, r_el, r_wri))

        # show
        cv2.putText(frame, f"Frame: {frame_idx}", (20,30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,255), 2)
        cv2.imshow(window, frame)
        if cv2.waitKey(1) & 0xFF == 27:
            break

        frame_idx += 1

    cap.release()
    cv2.destroyAllWindows()

    # -------------------------------
    # PREPROCESS (smooth & align)
    # -------------------------------
    l_ank_y = smooth(l_ank_y); r_ank_y = smooth(r_ank_y)
    l_knee_y = smooth(l_knee_y); r_knee_y = smooth(r_knee_y)

    n = min(len(l_ank_y), len(r_ank_y), len(l_knee_y), len(r_knee_y))
    if n < 12:
        return 1,1,1,1  # insufficient

    l_ank_y, r_ank_y = l_ank_y[:n], r_ank_y[:n]
    l_knee_y, r_knee_y = l_knee_y[:n], r_knee_y[:n]
    l_el_y, r_el_y = l_el_y[:n], r_el_y[:n]
    l_sh_y, r_sh_y = l_sh_y[:n], r_sh_y[:n]
    l_el_ang, r_el_ang = l_el_ang[:n], r_el_ang[:n]

    # -------------------------------
    # STEP / GALLOP SEGMENTATION
    # (use trailing leg peaks)
    # -------------------------------
    peaks,_ = find_peaks(-l_ank_y, distance=6)
    peaks = peaks[peaks < n]
    gallop_count = len(peaks)
    # print(gallop_count)

    # -------------------------------
    # LEAD LEG DETECTION
    # lead = leg with earlier forward swing (knee ahead)
    # -------------------------------
    lead_seq = []
    for i in range(n):
        if l_knee_y[i] < r_knee_y[i]:
            lead_seq.append("L")
        else:
            lead_seq.append("R")

    # count switches
    switches = sum(lead_seq[i]!=lead_seq[i-1] for i in range(1,len(lead_seq)))

    # -------------------------------
    # RHYTHM (interval consistency)
    # -------------------------------
    if len(peaks) > 1:
        intervals = np.diff(peaks) / max(fps,1)
        rhythm_var = safe_std(intervals)
    else:
        rhythm_var = 1.0

    # -------------------------------
    # FLIGHT PHASE (velocity-based)
    # -------------------------------
    vel_l = np.diff(l_ank_y)
    vel_r = np.diff(r_ank_y)

    contact_l = (l_ank_y[1:] > np.percentile(l_ank_y,70)) & (np.abs(vel_l)<0.01)
    contact_r = (r_ank_y[1:] > np.percentile(r_ank_y,70)) & (np.abs(vel_r)<0.01)

    flight = ~(contact_l | contact_r)
    flight_ratio = safe_mean(flight)

    # -------------------------------
    # ARM POSITION (waist-level + bend)
    # -------------------------------
    # elbows near waist (shoulder y vs elbow y)
    arm_pos = []
    for i in range(n):
        cond_pos = (abs(l_el_y[i] - l_sh_y[i]) < 0.1) and (abs(r_el_y[i] - r_sh_y[i]) < 0.1)
        cond_bend = (60 < l_el_ang[i] < 120) and (60 < r_el_ang[i] < 120)
        arm_pos.append(1 if (cond_pos and cond_bend) else 0)

    arm_ratio = safe_mean(arm_pos)

    # =========================================================
    # SCORING
    # =========================================================

    # Lead Leg Consistency
    lead_score = lead_score_cal(switches)

    # Rhythm
    rhythm_score = rhythm_score_cal(gallop_count, rhythm_var)

    # Flight Phase
    flight_score = flight_score_cal(flight_ratio)

    # Arm Position
    arm_score = arm_score_cal(arm_ratio)
    

    print("------ GALLOPING RESULT ------")
    print("Lead Leg:", lead_score)
    print("Rhythm:", rhythm_score)
    print("Flight:", flight_score)
    print("Arm Position:", arm_score)

    final_score = (lead_score + rhythm_score + flight_score + arm_score)/4

    return final_score

In [10]:
path = "data/galloping.mp4"
print(galloping_test(path))

Left cone X: 61
Right cone X: 688
0.07 --- 0.81


C:\Users\nayan\AppData\Local\Programs\Python\Python310\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


------ GALLOPING RESULT ------
Lead Leg: 2
Rhythm: 5
Flight: 5
Arm Position: 1
3.25
